# UrbanShift Churn Watchlist Generation — AWS

This notebook generates the final `churn_watchlist.csv` from the trained XGBoost churn model.

- model artefacts are loaded from S3
- the latest customer snapshot is scored
- customers are ranked by churn probability
- the final watchlist is exported to S3

**Business output:** ranked customer accounts at risk of reducing business with UrbanShift in the next 60 days.

## 1. Install and import dependencies

In [1]:
%pip install -q awswrangler xgboost scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import tempfile

import awswrangler as wr
import boto3
import joblib
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

## 2. Configure S3 paths


In [3]:
# Input model table created by the feature-engineering pipeline
S3_MODEL_TABLE_PATH = "s3://urbanshift-data-1781521886-rjjrqj/feature_engineering/customer_churn_model_table.csv"

# Existing trained model artefacts from the modelling notebook
S3_MODELING_PREFIX = "s3://urbanshift-data-1781521886-rjjrqj/modeling/xgboost/"

# New output location for the final watchlist
S3_WATCHLIST_OUTPUT_PATH = "s3://urbanshift-data-1781521886-rjjrqj/modeling/xgboost/churn_watchlist.csv"

# Expected artefact paths
S3_MODEL_PATH = S3_MODELING_PREFIX + "xgboost_tuned_model.joblib"
S3_FEATURE_COLUMNS_PATH = S3_MODELING_PREFIX + "xgboost_tuned_feature_columns.json"
S3_METRICS_PATH = S3_MODELING_PREFIX + "xgboost_tuned_metrics.json"

print("Model table:", S3_MODEL_TABLE_PATH)
print("Model artefact:", S3_MODEL_PATH)
print("Feature columns:", S3_FEATURE_COLUMNS_PATH)
print("Metrics:", S3_METRICS_PATH)
print("Watchlist output:", S3_WATCHLIST_OUTPUT_PATH)

Model table: s3://urbanshift-data-1781521886-rjjrqj/feature_engineering/customer_churn_model_table.csv
Model artefact: s3://urbanshift-data-1781521886-rjjrqj/modeling/xgboost/xgboost_tuned_model.joblib
Feature columns: s3://urbanshift-data-1781521886-rjjrqj/modeling/xgboost/xgboost_tuned_feature_columns.json
Metrics: s3://urbanshift-data-1781521886-rjjrqj/modeling/xgboost/xgboost_tuned_metrics.json
Watchlist output: s3://urbanshift-data-1781521886-rjjrqj/modeling/xgboost/churn_watchlist.csv


## 3. Helper functions for S3 artefacts

In [4]:
s3 = boto3.client("s3")


def parse_s3_uri(s3_uri: str) -> tuple[str, str]:
    # Split an S3 URI into bucket and key.
    if not s3_uri.startswith("s3://"):
        raise ValueError(f"Expected an S3 URI, got: {s3_uri}")
    bucket_key = s3_uri.replace("s3://", "", 1)
    bucket, key = bucket_key.split("/", 1)
    return bucket, key


def read_json_from_s3(s3_uri: str):
    # Read a JSON object from S3.
    bucket, key = parse_s3_uri(s3_uri)
    response = s3.get_object(Bucket=bucket, Key=key)
    return json.loads(response["Body"].read().decode("utf-8"))


def load_joblib_from_s3(s3_uri: str):
    # Download a joblib model from S3 to a temporary file, then load it.
    bucket, key = parse_s3_uri(s3_uri)
    with tempfile.NamedTemporaryFile(suffix=".joblib") as tmp:
        s3.download_file(bucket, key, tmp.name)
        return joblib.load(tmp.name)

## 4. Load trained model, feature columns, threshold, and model table

In [5]:
model = load_joblib_from_s3(S3_MODEL_PATH)
feature_columns = read_json_from_s3(S3_FEATURE_COLUMNS_PATH)
metrics = read_json_from_s3(S3_METRICS_PATH)

selected_threshold = float(metrics.get("selected_threshold", 0.5))

df = wr.s3.read_csv(S3_MODEL_TABLE_PATH)

print("Model loaded successfully")
print("Feature count:", len(feature_columns))
print("Selected threshold:", selected_threshold)
print("Model table shape:", df.shape)
display(df.head())

Model loaded successfully
Feature count: 29
Selected threshold: 0.7
Model table shape: (714, 22)


,customer_id,snapshot_date,customer_size,city,industry,industry_revenue_share,industry_relative_revenue_strength,payment_terms_days,customer_tenure_days,avg_revenue_per_delivery_lookback_60d,relative_volume_strength,volume_growth_lookback,relative_volume_growth_lookback,failed_delivery_rate_lookback_60d,return_rate_lookback_60d,avg_delivery_time_lookback_60d,incident_rate_lookback_60d,complaint_rate_lookback_60d,late_delivery_rate_lookback_60d,damaged_parcel_rate_lookback_60d,lost_parcel_rate_lookback_60d,at_risk
0,CUST1000,2024-11-30,Mid-size Retailer,London,Beauty,0.123535,0.0,60,901,5.948517,2.484211,-0.081301,-0.146712,0.042373,0.016949,72.724576,0.169492,0.038136,0.016949,0.046610,0.012712,0
1,CUST1001,2024-11-30,Mid-size Retailer,London,Books,0.088978,0.0,30,199,6.066206,3.884211,-0.016129,-0.081540,0.037940,0.027100,70.051491,0.176152,0.029810,0.032520,0.040650,0.016260,0
2,CUST1002,2024-11-30,Small Retailer,Bristol,Fashion,0.363793,0.0,30,759,4.657647,0.536842,-0.038462,-0.103873,0.078431,0.058824,65.039216,0.294118,0.039216,0.098039,0.058824,0.058824,0
3,CUST1003,2024-11-30,Small Retailer,Leeds,Beauty,0.123535,0.0,60,155,4.654000,0.368421,-0.157895,-0.223306,0.028571,0.057143,66.514286,0.200000,0.057143,0.057143,0.028571,0.000000,0
4,CUST1004,2024-11-30,Small Retailer,Manchester,Electronics,0.128233,0.0,30,886,4.626983,1.221053,-0.066667,-0.132078,0.051724,0.034483,66.025862,0.284483,0.034483,0.068966,0.068966,0.025862,0


## 5. Validate required columns

In [6]:
TARGET_COL = "at_risk"
TRACEABILITY_COLS = ["customer_id", "snapshot_date"]

missing_traceability = [col for col in TRACEABILITY_COLS if col not in df.columns]
if missing_traceability:
    raise ValueError(f"Missing required traceability columns: {missing_traceability}")

print("Traceability columns present:", TRACEABILITY_COLS)

if TARGET_COL in df.columns:
    print("Target column present. It will be excluded from scoring features.")
else:
    print("Target column not present. Proceeding with scoring-only data.")

Traceability columns present: ['customer_id', 'snapshot_date']
Target column present. It will be excluded from scoring features.


## 6. Recreate the feature matrix in the same structure used during training

The model was trained using `pandas.get_dummies()` and the saved `feature_columns` list. This cell rebuilds the encoded matrix and aligns the columns exactly to the training order.

In [7]:
drop_cols = TRACEABILITY_COLS.copy()
if TARGET_COL in df.columns:
    drop_cols.append(TARGET_COL)

X_raw = df.drop(columns=drop_cols)

categorical_cols = X_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
X_encoded = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True, dtype=int)

# Align scoring data to the exact training feature list.
# Missing dummy columns are created as 0. Extra columns are discarded.
X_score = X_encoded.reindex(columns=feature_columns, fill_value=0)

print("Raw feature shape:", X_raw.shape)
print("Encoded feature shape before alignment:", X_encoded.shape)
print("Scoring feature shape after alignment:", X_score.shape)

if list(X_score.columns) != list(feature_columns):
    raise ValueError("Scoring feature columns do not match saved training feature columns.")

print("Feature alignment passed.")

Raw feature shape: (714, 19)
Encoded feature shape before alignment: (714, 29)
Scoring feature shape after alignment: (714, 29)
Feature alignment passed.


## 7. Score all customer snapshots

In [8]:
scored_df = df.copy()

scored_df["churn_probability"] = model.predict_proba(X_score)[:, 1]
scored_df["predicted_at_risk"] = (
    scored_df["churn_probability"] >= selected_threshold
).astype(int)

print(f"Rows scored: {len(scored_df):,}")
display(scored_df[["customer_id", "snapshot_date", "churn_probability", "predicted_at_risk"]].head())

Rows scored: 714


,customer_id,snapshot_date,churn_probability,predicted_at_risk
0,CUST1000,2024-11-30,0.101781,0
1,CUST1001,2024-11-30,0.004535,0
2,CUST1002,2024-11-30,0.053447,0
3,CUST1003,2024-11-30,0.210436,0
4,CUST1004,2024-11-30,0.019963,0


## 8. Keep only the latest snapshot per customer

The model table contains monthly customer snapshots. For the final executive watchlist, each customer should appear once using their most recent available snapshot.

In [9]:
scored_df["snapshot_date"] = pd.to_datetime(scored_df["snapshot_date"])
latest_snapshot = scored_df["snapshot_date"].max()

latest_scored_df = (
    scored_df
    .loc[scored_df["snapshot_date"] == latest_snapshot]
    .copy()
)

print("Latest snapshot:", latest_snapshot.date())
print(f"Customers in latest snapshot: {len(latest_scored_df):,}")

Latest snapshot: 2025-04-30
Customers in latest snapshot: 117


## 9. Add risk levels and customer rank

In [10]:
def assign_risk_level(probability: float) -> str:
    # Convert churn probability into an executive-friendly risk band.
    if probability >= 0.80:
        return "High"
    if probability >= 0.50:
        return "Medium"
    return "Low"

latest_scored_df["risk_level"] = latest_scored_df["churn_probability"].apply(assign_risk_level)

latest_scored_df = (
    latest_scored_df
    .sort_values("churn_probability", ascending=False)
    .reset_index(drop=True)
)

latest_scored_df["risk_rank"] = latest_scored_df.index + 1

display(latest_scored_df[["risk_rank", "customer_id", "snapshot_date", "churn_probability", "risk_level"]].head(20))

,risk_rank,customer_id,snapshot_date,churn_probability,risk_level
0,1,CUST1000,2025-04-30,0.981921,High
1,2,CUST1044,2025-04-30,0.973798,High
2,3,CUST1010,2025-04-30,0.969705,High
3,4,CUST1047,2025-04-30,0.960642,High
4,5,CUST1109,2025-04-30,0.959085,High
5,6,CUST1073,2025-04-30,0.958797,High
6,7,CUST1055,2025-04-30,0.957801,High
7,8,CUST1091,2025-04-30,0.955726,High
8,9,CUST1040,2025-04-30,0.948985,High
9,10,CUST1004,2025-04-30,0.948254,High


## 10. Build the final executive watchlist

Only available business context columns are included, so the cell remains robust if the modelling table changes slightly.

In [11]:
preferred_columns = [
    "risk_rank",
    "customer_id",
    "customer_size",
    "industry",
    "city",
    "churn_probability",
    "risk_level",
    "predicted_at_risk",
    "snapshot_date",
]

available_columns = [col for col in preferred_columns if col in latest_scored_df.columns]

churn_watchlist = latest_scored_df[available_columns].copy()

# Round probability for readability while preserving enough precision for ranking.
churn_watchlist["churn_probability"] = churn_watchlist["churn_probability"].round(4)

print(f"Final watchlist rows: {len(churn_watchlist):,}")
display(churn_watchlist.head(20))

Final watchlist rows: 117


,risk_rank,customer_id,customer_size,industry,city,churn_probability,risk_level,predicted_at_risk,snapshot_date
0,1,CUST1000,Mid-size Retailer,Beauty,London,0.9819,High,1,2025-04-30
1,2,CUST1044,Small Retailer,Fashion,London,0.9738,High,1,2025-04-30
2,3,CUST1010,Small Retailer,Electronics,Bristol,0.9697,High,1,2025-04-30
3,4,CUST1047,Mid-size Retailer,Home,Glasgow,0.9606,High,1,2025-04-30
4,5,CUST1109,Small Retailer,Home,London,0.9591,High,1,2025-04-30
5,6,CUST1073,Small Retailer,Food & Drink,Glasgow,0.9588,High,1,2025-04-30
6,7,CUST1055,Mid-size Retailer,Electronics,London,0.9578,High,1,2025-04-30
7,8,CUST1091,Mid-size Retailer,Home,Glasgow,0.9557,High,1,2025-04-30
8,9,CUST1040,Small Retailer,Beauty,Birmingham,0.9490,High,1,2025-04-30
9,10,CUST1004,Small Retailer,Electronics,Manchester,0.9483,High,1,2025-04-30


## 11. Watchlist summary for reporting

In [12]:
risk_counts = (
    churn_watchlist["risk_level"]
    .value_counts()
    .rename_axis("risk_level")
    .reset_index(name="customers")
)

display(risk_counts)

display(
    churn_watchlist
    .groupby("risk_level", observed=True)["churn_probability"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean", ascending=False)
)

,risk_level,customers
0,Low,98
1,High,15
2,Medium,4


,count,mean,min,max
risk_level,,,,
High,15,0.938780,0.8021,0.9819
Medium,4,0.556800,0.5106,0.6027
Low,98,0.052069,0.0009,0.3398


## 12. Top 20 highest-risk accounts

This table is suitable for the presentation deck or account-manager follow-up.

In [13]:
top20_watchlist = churn_watchlist.head(20).copy()
display(top20_watchlist)

,risk_rank,customer_id,customer_size,industry,city,churn_probability,risk_level,predicted_at_risk,snapshot_date
0,1,CUST1000,Mid-size Retailer,Beauty,London,0.9819,High,1,2025-04-30
1,2,CUST1044,Small Retailer,Fashion,London,0.9738,High,1,2025-04-30
2,3,CUST1010,Small Retailer,Electronics,Bristol,0.9697,High,1,2025-04-30
3,4,CUST1047,Mid-size Retailer,Home,Glasgow,0.9606,High,1,2025-04-30
4,5,CUST1109,Small Retailer,Home,London,0.9591,High,1,2025-04-30
5,6,CUST1073,Small Retailer,Food & Drink,Glasgow,0.9588,High,1,2025-04-30
6,7,CUST1055,Mid-size Retailer,Electronics,London,0.9578,High,1,2025-04-30
7,8,CUST1091,Mid-size Retailer,Home,Glasgow,0.9557,High,1,2025-04-30
8,9,CUST1040,Small Retailer,Beauty,Birmingham,0.9490,High,1,2025-04-30
9,10,CUST1004,Small Retailer,Electronics,Manchester,0.9483,High,1,2025-04-30


## 13. Export `churn_watchlist.csv` to S3

In [14]:
wr.s3.to_csv(
    df=churn_watchlist,
    path=S3_WATCHLIST_OUTPUT_PATH,
    index=False,
)

print(f"Exported churn watchlist to: {S3_WATCHLIST_OUTPUT_PATH}")

Exported churn watchlist to: s3://urbanshift-data-1781521886-rjjrqj/modeling/xgboost/churn_watchlist.csv


## Notes for interpretation

- This notebook does **not** train or tune a model.
- The trained XGBoost model, selected threshold, and feature-column list are loaded from S3.
- The watchlist uses the latest available customer snapshot only.
- `churn_probability` is a model score, not proof that a customer will churn.
- `risk_level` is a simple executive-facing banding rule used to support prioritisation.